# GameTheory-11b — Jeux Bayésiens en Lean 4 (companion)

**Notebook compagnon Lean 4 de `GameTheory-11-BayesianGames.ipynb`** (qui traite les jeux bayésiens en Python).
Cette extension formalise en Lean 4 le résultat central des enchères à valeur privée : **le théorème de Vickrey** (enchère au second prix — *second-price sealed-bid*), exécuté dans le kernel `lean4-wsl`.

## Provenance — ces résultats sont prouvés SANS sorry dans le lake

Les théorèmes exécutés ci-dessous sont **restatés à l'identique** depuis le lake
[`lean_game_defs_ext/`](https://github.com/jsboige/CoursIA/tree/main/MyIA.AI.Notebooks/GameTheory/lean_game_defs_ext)
(module `Bayesian`), où ils sont **prouvés sans aucun `sorry`**. Le lake est un
projet Lean 4 *self-contained* (core Lean 4, **sans Mathlib**) — donc ses énoncés
se ré-élaborent tels quels dans le kernel notebook. On les restate ici pour
**exécution pédagogique vérifiée** : le kernel `lean4-wsl` élabore réellement les
vrais énoncés et produire les sorties (`#check`, `#eval`) ci-dessous.

> **Preuve de compilation du lake (exécution réelle, hors notebook car le kernel ne lance pas `lake`) :**
> ```
> $ cd lean_game_defs_ext && lake build Bayesian
> ✔ [13/13] Built Bayesian
> Build completed successfully (13 jobs).
> $ grep -rnE '^\s*sorry\b' Bayesian/*.lean | wc -l
> 0
> ```

**Pourquoi restate inline plutôt que `import Bayesian.Vickrey` ?** Le kernel
interactif `lean4-wsl` (REPL `lean4_jupyter`) résout les `import` de modules lake
**uniquement au démarrage du process REPL**, pas depuis une commande `import`
interactive en cours de notebook (un `import` en milieu de flux n'est pas un
module-load en Lean). Aucun notebook `GameTheory/*.ipynb` n'importe un lake — les
notebooks Lean `2b/4b/8b/15b` sont tous *self-contained*. Ce notebook suit ce
même pattern établi (issue #4040, voie B). Le travail d'exploration
import-from-lake (prelude `NotebookCtx`, launch via lake-env) continue en
side-track sur #4251.


In [1]:
#eval 2 + 2
#check Nat


#eval 2 + 2
─────▶  4
#check Nat
──────▶  Nat : Type

--% env 0

Raw input:
{"cmd": "#eval 2 + 2\n#check Nat\n"}
Raw output:
{"messages":
 [{"severity": "info",
   "pos": {"line": 1, "column": 0},
   "endPos": {"line": 1, "column": 5},
   "data": "4"},
  {"severity": "info",
   "pos": {"line": 2, "column": 0},
   "endPos": {"line": 2, "column": 6},
   "data": "Nat : Type"}],
 "env": 0}

## 1. Sommes finies sur `Fin n` (`sumFin`)

Mathlib n'étant pas disponible (projet core-only), on définit une somme par récurrence structurelle sur `n`. C'est l'infrastructure d'espérance utilité des jeux bayésiens. (Source : `lean_game_defs_ext/Bayesian/Sum.lean`.)


In [2]:
def sumFin : (n : Nat) → (Fin n → Int) → Int
  | 0, _ => 0
  | n + 1, f => f 0 + sumFin n (fun i => f i.succ)

@[simp] theorem sumFin_zero (f : Fin 0 → Int) : sumFin 0 f = 0 := rfl

@[simp] theorem sumFin_succ (n : Nat) (f : Fin (n + 1) → Int) :
    sumFin (n + 1) f = f 0 + sumFin n (fun i => f i.succ) := rfl

@[simp] theorem sumFin_zero_fun (n : Nat) :
    sumFin n (fun _ => (0 : Int)) = 0 := by
  induction n with
  | zero => simp
  | succ n ih => simp [ih]

/-- La dominance ponctuelle se transfère aux sommes. -/
theorem sumFin_mono {n : Nat} {f g : Fin n → Int} (h : ∀ i, f i ≤ g i) :
    sumFin n f ≤ sumFin n g := by
  induction n with
  | zero => simp
  | succ n ih =>
    simp only [sumFin_succ]
    exact Int.add_le_add (h 0) (ih (fun i => h i.succ))


def sumFin : (n : Nat) → (Fin n → Int) → Int
  | 0, _ => 0
  | n + 1, f => f 0 + sumFin n (fun i => f i.succ)

@[simp] theorem sumFin_zero (f : Fin 0 → Int) : sumFin 0 f = 0 := rfl

@[simp] theorem sumFin_succ (n : Nat) (f : Fin (n + 1) → Int) :
    sumFin (n + 1) f = f 0 + sumFin n (fun i => f i.succ) := rfl

@[simp] theorem sumFin_zero_fun (n : Nat) :
    sumFin n (fun _ => (0 : Int)) = 0 := by
  induction n with
  | zero => simp
  | succ n ih => simp [ih]

/-- La dominance ponctuelle se transfère aux sommes. -/
theorem sumFin_mono {n : Nat} {f g : Fin n → Int} (h : ∀ i, f i ≤ g i) :
    sumFin n f ≤ sumFin n g := by
  induction n with
  | zero => simp
  | succ n ih =>
    simp only [sumFin_succ]
    exact Int.add_le_add (h 0) (ih (fun i => h i.succ))

--% env 1

Raw input:
{"cmd": "def sumFin : (n : Nat) \u2192 (Fin n \u2192 Int) \u2192 Int\n  | 0, _ => 0\n  | n + 1, f => f 0 + sumFin n (fun i => f i.succ)\n\n@[simp] theorem sumFin_zero (f : Fin 0 \u2192 Int) : sumFin 0 f = 0 := rfl\n\n@[simp] theorem sumFin_succ (n : Nat) (f : Fin (n + 1) \u2192 Int) :\n    sumFin (n + 1) f = f 0 + sumFin n (fun i => f i.succ) := rfl\n\n@[simp] theorem sumFin_zero_fun (n : Nat) :\n    sumFin n (fun _ => (0 : Int)) = 0 := by\n  induction n with\n  | zero => simp\n  | succ n ih => simp [ih]\n\n/-- La dominance ponctuelle se transf\u00e8re aux sommes. -/\ntheorem sumFin_mono {n : Nat} {f g : Fin n \u2192 Int} (h : \u2200 i, f i \u2264 g i) :\n    sumFin n f \u2264 sumFin n g := by\n  induction n with\n  | zero => simp\n  | succ n ih =>\n    simp only [sumFin_succ]\n    exact Int.add_le_add (h 0) (ih (fun i => h i.succ))\n", "env": 0}
Raw output:
{"env": 1}

## 2. Types d'Harsanyi — jeu bayésien fini à deux joueurs

`BayesGame2` encode un jeu bayésien à deux joueurs, types et actions finis (`Fin n`), avec un *prior commun non normalisé* donné par des poids `Nat` (`w t1 t2`). Les utilités dépendent du profil de types complet et du profil d'actions. Les stratégies sont *pures et contingentes au type* ; `interimU` est l'utilité espérée interim (conditionnelle à son propre type). (Source : `lean_game_defs_ext/Bayesian/Types.lean`.)


In [3]:
structure BayesGame2 where
  nT1 : Nat
  nT2 : Nat
  nA1 : Nat
  nA2 : Nat
  w : Fin nT1 → Fin nT2 → Nat
  u1 : Fin nT1 → Fin nT2 → Fin nA1 → Fin nA2 → Int
  u2 : Fin nT1 → Fin nT2 → Fin nA1 → Fin nA2 → Int

def Strategy1 (g : BayesGame2) := Fin g.nT1 → Fin g.nA1
def Strategy2 (g : BayesGame2) := Fin g.nT2 → Fin g.nA2

def interimU1 (g : BayesGame2) (t1 : Fin g.nT1) (a : Fin g.nA1)
    (s2 : Strategy2 g) : Int :=
  sumFin g.nT2 (fun t2 => (g.w t1 t2 : Int) * g.u1 t1 t2 a (s2 t2))

def interimU2 (g : BayesGame2) (t2 : Fin g.nT2) (a : Fin g.nA2)
    (s1 : Strategy1 g) : Int :=
  sumFin g.nT1 (fun t1 => (g.w t1 t2 : Int) * g.u2 t1 t2 (s1 t1) a)

def exAnteU1 (g : BayesGame2) (s1 : Strategy1 g) (s2 : Strategy2 g) : Int :=
  sumFin g.nT1 (fun t1 => interimU1 g t1 (s1 t1) s2)

def exAnteU2 (g : BayesGame2) (s1 : Strategy1 g) (s2 : Strategy2 g) : Int :=
  sumFin g.nT2 (fun t2 => interimU2 g t2 (s2 t2) s1)


structure BayesGame2 where
  nT1 : Nat
  nT2 : Nat
  nA1 : Nat
  nA2 : Nat
  w : Fin nT1 → Fin nT2 → Nat
  u1 : Fin nT1 → Fin nT2 → Fin nA1 → Fin nA2 → Int
  u2 : Fin nT1 → Fin nT2 → Fin nA1 → Fin nA2 → Int

def Strategy1 (g : BayesGame2) := Fin g.nT1 → Fin g.nA1
def Strategy2 (g : BayesGame2) := Fin g.nT2 → Fin g.nA2

def interimU1 (g : BayesGame2) (t1 : Fin g.nT1) (a : Fin g.nA1)
    (s2 : Strategy2 g) : Int :=
  sumFin g.nT2 (fun t2 => (g.w t1 t2 : Int) * g.u1 t1 t2 a (s2 t2))

def interimU2 (g : BayesGame2) (t2 : Fin g.nT2) (a : Fin g.nA2)
    (s1 : Strategy1 g) : Int :=
  sumFin g.nT1 (fun t1 => (g.w t1 t2 : Int) * g.u2 t1 t2 (s1 t1) a)

def exAnteU1 (g : BayesGame2) (s1 : Strategy1 g) (s2 : Strategy2 g) : Int :=
  sumFin g.nT1 (fun t1 => interimU1 g t1 (s1 t1) s2)

def exAnteU2 (g : BayesGame2) (s1 : Strategy1 g) (s2 : Strategy2 g) : Int :=
  sumFin g.nT2 (fun t2 => interimU2 g t2 (s2 t2) s1)

--% env 2

Raw input:
{"cmd": "structure BayesGame2 where\n  nT1 : Nat\n  nT2 : Nat\n  nA1 : Nat\n  nA2 : Nat\n  w : Fin nT1 \u2192 Fin nT2 \u2192 Nat\n  u1 : Fin nT1 \u2192 Fin nT2 \u2192 Fin nA1 \u2192 Fin nA2 \u2192 Int\n  u2 : Fin nT1 \u2192 Fin nT2 \u2192 Fin nA1 \u2192 Fin nA2 \u2192 Int\n\ndef Strategy1 (g : BayesGame2) := Fin g.nT1 \u2192 Fin g.nA1\ndef Strategy2 (g : BayesGame2) := Fin g.nT2 \u2192 Fin g.nA2\n\ndef interimU1 (g : BayesGame2) (t1 : Fin g.nT1) (a : Fin g.nA1)\n    (s2 : Strategy2 g) : Int :=\n  sumFin g.nT2 (fun t2 => (g.w t1 t2 : Int) * g.u1 t1 t2 a (s2 t2))\n\ndef interimU2 (g : BayesGame2) (t2 : Fin g.nT2) (a : Fin g.nA2)\n    (s1 : Strategy1 g) : Int :=\n  sumFin g.nT1 (fun t1 => (g.w t1 t2 : Int) * g.u2 t1 t2 (s1 t1) a)\n\ndef exAnteU1 (g : BayesGame2) (s1 : Strategy1 g) (s2 : Strategy2 g) : Int :=\n  sumFin g.nT1 (fun t1 => interimU1 g t1 (s1 t1) s2)\n\ndef exAnteU2 (g : BayesGame2) (s1 : Strategy1 g) (s2 : Strategy2 g) : Int :=\n  sumFin g.nT2 (fun t2 => interimU2 g t2 (s2 t2) s1)\n", "env": 1}
Raw output:
{"env": 2}

## 3. Équilibre Bayésien de Nash (`isBNE`)

Un profil de stratégies est un **BNE** si aucun type d'aucun joueur n'a de déviante d'action unilatérale profitable, en utilité interim. Les quantificateurs ne portent que sur des types `Fin` et des actions `Fin` de comparaisons `Int` décidables → **la propriété est décidable** (`decide` fonctionne sur les jeux concrets). (Source : `lean_game_defs_ext/Bayesian/BNE.lean`.)


In [4]:
@[reducible] def isBNE (g : BayesGame2) (s1 : Strategy1 g) (s2 : Strategy2 g) : Prop :=
  (∀ t1 : Fin g.nT1, ∀ a : Fin g.nA1,
      interimU1 g t1 a s2 ≤ interimU1 g t1 (s1 t1) s2) ∧
  (∀ t2 : Fin g.nT2, ∀ a : Fin g.nA2,
      interimU2 g t2 a s1 ≤ interimU2 g t2 (s2 t2) s1)


@[reducible] def isBNE (g : BayesGame2) (s1 : Strategy1 g) (s2 : Strategy2 g) : Prop :=
  (∀ t1 : Fin g.nT1, ∀ a : Fin g.nA1,
      interimU1 g t1 a s2 ≤ interimU1 g t1 (s1 t1) s2) ∧
  (∀ t2 : Fin g.nT2, ∀ a : Fin g.nA2,
      interimU2 g t2 a s1 ≤ interimU2 g t2 (s2 t2) s1)

--% env 3

Raw input:
{"cmd": "@[reducible] def isBNE (g : BayesGame2) (s1 : Strategy1 g) (s2 : Strategy2 g) : Prop :=\n  (\u2200 t1 : Fin g.nT1, \u2200 a : Fin g.nA1,\n      interimU1 g t1 a s2 \u2264 interimU1 g t1 (s1 t1) s2) \u2227\n  (\u2200 t2 : Fin g.nT2, \u2200 a : Fin g.nA2,\n      interimU2 g t2 a s1 \u2264 interimU2 g t2 (s2 t2) s1)\n", "env": 2}
Raw output:
{"env": 3}

## 4. Enchère de Vickrey (second-prix) — l'enchère `spa`

Deux enchérisseurs, valorisations et enchères dans `Fin n`, prior uniforme (poids 1). Le gagnant paie l'enchère du perdant (le *second prix*) ; les utilités sont mises à l'échelle par 2 pour que le partage d'égalité reste entier. Enchérer sa vraie valeur (`spaTruthful`) est la stratégie étudiée. (Source : `lean_game_defs_ext/Bayesian/Vickrey.lean`.)


In [5]:
@[reducible] def spa (n : Nat) : BayesGame2 where
  nT1 := n
  nT2 := n
  nA1 := n
  nA2 := n
  w := fun _ _ => 1
  u1 := fun v1 _ b1 b2 =>
    if b2.val < b1.val then 2 * ((v1.val : Int) - b2.val)
    else if b1.val = b2.val then (v1.val : Int) - b2.val
    else 0
  u2 := fun _ v2 b1 b2 =>
    if b1.val < b2.val then 2 * ((v2.val : Int) - b1.val)
    else if b1.val = b2.val then (v2.val : Int) - b1.val
    else 0

/-- Enchère honnête (joueur 1) : bid = valeur. -/
@[reducible] def spaTruthful1 (n : Nat) : Strategy1 (spa n) := fun v => v

/-- Enchère honnête (joueur 2). -/
@[reducible] def spaTruthful2 (n : Nat) : Strategy2 (spa n) := fun v => v


@[reducible] def spa (n : Nat) : BayesGame2 where
  nT1 := n
  nT2 := n
  nA1 := n
  nA2 := n
  w := fun _ _ => 1
  u1 := fun v1 _ b1 b2 =>
    if b2.val < b1.val then 2 * ((v1.val : Int) - b2.val)
    else if b1.val = b2.val then (v1.val : Int) - b2.val
    else 0
  u2 := fun _ v2 b1 b2 =>
    if b1.val < b2.val then 2 * ((v2.val : Int) - b1.val)
    else if b1.val = b2.val then (v2.val : Int) - b1.val
    else 0

/-- Enchère honnête (joueur 1) : bid = valeur. -/
@[reducible] def spaTruthful1 (n : Nat) : Strategy1 (spa n) := fun v => v

/-- Enchère honnête (joueur 2). -/
@[reducible] def spaTruthful2 (n : Nat) : Strategy2 (spa n) := fun v => v

--% env 4

Raw input:
{"cmd": "@[reducible] def spa (n : Nat) : BayesGame2 where\n  nT1 := n\n  nT2 := n\n  nA1 := n\n  nA2 := n\n  w := fun _ _ => 1\n  u1 := fun v1 _ b1 b2 =>\n    if b2.val < b1.val then 2 * ((v1.val : Int) - b2.val)\n    else if b1.val = b2.val then (v1.val : Int) - b2.val\n    else 0\n  u2 := fun _ v2 b1 b2 =>\n    if b1.val < b2.val then 2 * ((v2.val : Int) - b1.val)\n    else if b1.val = b2.val then (v2.val : Int) - b1.val\n    else 0\n\n/-- Ench\u00e8re honn\u00eate (joueur 1) : bid = valeur. -/\n@[reducible] def spaTruthful1 (n : Nat) : Strategy1 (spa n) := fun v => v\n\n/-- Ench\u00e8re honn\u00eate (joueur 2). -/\n@[reducible] def spaTruthful2 (n : Nat) : Strategy2 (spa n) := fun v => v\n", "env": 3}
Raw output:
{"env": 4}

### 4.1 Argument de dominance

L'argument classique de Vickrey : enchérir sa vraie valeur **domine faiblement** toute autre enchère, *point par point* — contre n'importe quelle enchère adverse, dans n'importe quel état. Quel que soit le prix (l'enchère adverse), dévier de `v` ne peut que forfaiter un gain profitable (`b1 < v`) ou acheter à perte (`b1 > v`), jamais améliorer — car le **prix payé ne dépend pas de sa propre enchère**. La dominance ponctuelle se transfère ensuite à l'utilité interim via `sumFin_mono`.


In [6]:
/-- Vickrey dominance, joueur 1 : la vraie valeur domine faiblement toute autre enchère.
    On déplie `spa` explicitement via `simp only [spa]` (au lieu d'un `show` defeq)
    pour rester robuste aux variations d'élaboration defeq entre versions de Lean ;
    la preuve est identique à celle du lake (case analysis sur qui gagne, close par `omega`).
    NB : le linter signale « simp argument unused: spa » — faux positif connu :
    `simp` déplie effectivement `spa` (nécessaire pour que `split` voie les `if`),
    mais ne le compte pas comme règle de réécriture. -/
theorem spa_truthful_dominant1 (n : Nat) (v t2 b1 b2 : Fin n) :
    (spa n).u1 v t2 b1 b2 ≤ (spa n).u1 v t2 v b2 := by
  simp only [spa]
  repeat' split
  all_goals omega

/-- Vickrey dominance, joueur 2 (symétrique). -/
theorem spa_truthful_dominant2 (n : Nat) (t1 v b1 b2 : Fin n) :
    (spa n).u2 t1 v b1 b2 ≤ (spa n).u2 t1 v b1 v := by
  simp only [spa]
  repeat' split
  all_goals omega

/-- La dominance ponctuelle => meilleure réponse interim (joueur 1). -/
theorem spa_interim_best1 (n : Nat) (v a : Fin n) (s2 : Strategy2 (spa n)) :
    interimU1 (spa n) v a s2 ≤ interimU1 (spa n) v (spaTruthful1 n v) s2 := by
  apply sumFin_mono
  intro t2
  exact Int.mul_le_mul_of_nonneg_left
    (spa_truthful_dominant1 n v t2 a (s2 t2)) (by omega)

/-- Meilleure réponse interim, joueur 2. -/
theorem spa_interim_best2 (n : Nat) (v a : Fin n) (s1 : Strategy1 (spa n)) :
    interimU2 (spa n) v a s1 ≤ interimU2 (spa n) v (spaTruthful2 n v) s1 := by
  apply sumFin_mono
  intro t1
  exact Int.mul_le_mul_of_nonneg_left
    (spa_truthful_dominant2 n t1 v (s1 t1) a) (by omega)


/-- Vickrey dominance, joueur 1 : la vraie valeur domine faiblement toute autre enchère.
    On déplie `spa` explicitement via `simp only [spa]` (au lieu d'un `show` defeq)
    pour rester robuste aux variations d'élaboration defeq entre versions de Lean ;
    la preuve est identique à celle du lake (case analysis sur qui gagne, close par `omega`).
    NB : le linter signale « simp argument unused: spa » — faux positif connu :
    `simp` déplie effectivement `spa` (nécessaire pour que `split` voie les `if`),
    mais ne le compte pas comme règle de réécriture. -/
theorem spa_truthful_dominant1 (n : Nat) (v t2 b1 b2 : Fin n) :
    (spa n).u1 v t2 b1 b2 ≤ (spa n).u1 v t2 v b2 := by
  simp only [spa]
             ───▶ 🟨 This simp argument is unused:
  spa

Hint: Omit it from the simp argument list.
  simp only ̵[̵s̵p̵a̵]̵

Note: This linter can be disabled with `set_option linter.unusedSimpArgs false`
  repeat' split
  all_goals omega

/-- Vickrey dominance, joueur 2 (symétrique). -/
theorem spa_truthful_dominant2 (n : Nat) (t1 v b1 b2 : Fin n) :
    (spa n).u2 t1 v b1 b2 ≤ (spa n).u2 t1 v b1 v := by
  simp only [spa]
             ───▶ 🟨 This simp argument is unused:
  spa

Hint: Omit it from the simp argument list.
  simp only ̵[̵s̵p̵a̵]̵

Note: This linter can be disabled with `set_option linter.unusedSimpArgs false`
  repeat' split
  all_goals omega

/-- La dominance ponctuelle => meilleure réponse interim (joueur 1). -/
theorem spa_interim_best1 (n : Nat) (v a : Fin n) (s2 : Strategy2 (spa n)) :
    interimU1 (spa n) v a s2 ≤ interimU1 (spa n) v (spaTruthful1 n v) s2 := by
  apply sumFin_mono
  intro t2
  exact Int.mul_le_mul_of_nonneg_left
    (spa_truthful_dominant1 n v t2 a (s2 t2)) (by omega)

/-- Meilleure réponse interim, joueur 2. -/
theorem spa_interim_best2 (n : Nat) (v a : Fin n) (s1 : Strategy1 (spa n)) :
    interimU2 (spa n) v a s1 ≤ interimU2 (spa n) v (spaTruthful2 n v) s1 := by
  apply sumFin_mono
  intro t1
  exact Int.mul_le_mul_of_nonneg_left
    (spa_truthful_dominant2 n t1 v (s1 t1) a) (by omega)

--% env 5

Raw input:
{"cmd": "/-- Vickrey dominance, joueur 1 : la vraie valeur domine faiblement toute autre ench\u00e8re.\n    On d\u00e9plie `spa` explicitement via `simp only [spa]` (au lieu d'un `show` defeq)\n    pour rester robuste aux variations d'\u00e9laboration defeq entre versions de Lean ;\n    la preuve est identique \u00e0 celle du lake (case analysis sur qui gagne, close par `omega`).\n    NB : le linter signale \u00ab simp argument unused: spa \u00bb \u2014 faux positif connu :\n    `simp` d\u00e9plie effectivement `spa` (n\u00e9cessaire pour que `split` voie les `if`),\n    mais ne le compte pas comme r\u00e8gle de r\u00e9\u00e9criture. -/\ntheorem spa_truthful_dominant1 (n : Nat) (v t2 b1 b2 : Fin n) :\n    (spa n).u1 v t2 b1 b2 \u2264 (spa n).u1 v t2 v b2 := by\n  simp only [spa]\n  repeat' split\n  all_goals omega\n\n/-- Vickrey dominance, joueur 2 (sym\u00e9trique). -/\ntheorem spa_truthful_dominant2 (n : Nat) (t1 v b1 b2 : Fin n) :\n    (spa n).u2 t1 v b1 b2 \u2264 (spa n).u2 t1 v b1 v := by\n  simp only [spa]\n  repeat' split\n  all_goals omega\n\n/-- La dominance ponctuelle => meilleure r\u00e9ponse interim (joueur 1). -/\ntheorem spa_interim_best1 (n : Nat) (v a : Fin n) (s2 : Strategy2 (spa n)) :\n    interimU1 (spa n) v a s2 \u2264 interimU1 (spa n) v (spaTruthful1 n v) s2 := by\n  apply sumFin_mono\n  intro t2\n  exact Int.mul_le_mul_of_nonneg_left\n    (spa_truthful_dominant1 n v t2 a (s2 t2)) (by omega)\n\n/-- Meilleure r\u00e9ponse interim, joueur 2. -/\ntheorem spa_interim_best2 (n : Nat) (v a : Fin n) (s1 : Strategy1 (spa n)) :\n    interimU2 (spa n) v a s1 \u2264 interimU2 (spa n) v (spaTruthful2 n v) s1 := by\n  apply sumFin_mono\n  intro t1\n  exact Int.mul_le_mul_of_nonneg_left\n    (spa_truthful_dominant2 n t1 v (s1 t1) a) (by omega)\n", "env": 4}
Raw output:
{"messages":
 [{"severity": "warning",
   "pos": {"line": 10, "column": 13},
   "endPos": {"line": 10, "column":

### 4.2 Théorème de Vickrey — résultat clé

**Le profil d'enchères honnêtes est un équilibre bayésien de Nash de l'enchère au second prix, pour TOUT `n`.** Théorème général — sans `decide`, sans vérification d'instance — ce qui contraste avec l'enchère au premier prix, où l'enchère honnête n'est PAS un BNE (cf. `Auction.lean` dans le lake). La preuve combine les deux meilleures réponses interim ci-dessus dans la conjonction `isBNE`.


In [7]:
/-- **Théorème de Vickrey** (fini, deux enchérisseurs) : le profil honnête est un BNE pour tout n. -/
theorem spa_truthful_bne (n : Nat) :
    isBNE (spa n) (spaTruthful1 n) (spaTruthful2 n) :=
  ⟨fun t1 a => spa_interim_best1 n t1 a (spaTruthful2 n),
   fun t2 a => spa_interim_best2 n t2 a (spaTruthful1 n)⟩

#check spa_truthful_bne


/-- **Théorème de Vickrey** (fini, deux enchérisseurs) : le profil honnête est un BNE pour tout n. -/
theorem spa_truthful_bne (n : Nat) :
    isBNE (spa n) (spaTruthful1 n) (spaTruthful2 n) :=
  ⟨fun t1 a => spa_interim_best1 n t1 a (spaTruthful2 n),
   fun t2 a => spa_interim_best2 n t2 a (spaTruthful1 n)⟩

#check spa_truthful_bne
──────▶  spa_truthful_bne (n : Nat) : isBNE (spa n) (spaTruthful1 n) (spaTruthful2 n)

--% env 6

Raw input:
{"cmd": "/-- **Th\u00e9or\u00e8me de Vickrey** (fini, deux ench\u00e9risseurs) : le profil honn\u00eate est un BNE pour tout n. -/\ntheorem spa_truthful_bne (n : Nat) :\n    isBNE (spa n) (spaTruthful1 n) (spaTruthful2 n) :=\n  \u27e8fun t1 a => spa_interim_best1 n t1 a (spaTruthful2 n),\n   fun t2 a => spa_interim_best2 n t2 a (spaTruthful1 n)\u27e9\n\n#check spa_truthful_bne\n", "env": 5}
Raw output:
{"messages":
 [{"severity": "info",
   "pos": {"line": 7, "column": 0},
   "endPos": {"line": 7, "column": 6},
   "data":
   "spa_truthful_bne (n : Nat) : isBNE (spa n) (spaTruthful1 n) (spaTruthful2 n)"}],
 "env": 6}

### 4.3 Rente d'information — instance décidable

À l'équilibre honnête Vickrey avec valorisations `{0, 1, 2}`, le type haut gagne une utilité interim **strictement positive** (rente d'information concrète = 6 à l'échelle : gagne en payant 0 puis 1 contre les types bas, égalité à 2 pour un surplus nul). La décidabilité de `isBNE` (quantificateurs sur `Fin` de comparaisons `Int`) permet de le prouver par `decide` sur cette instance concrète.


In [8]:
/-- Le type haut garde une rente d'information strictement positive (instance concrète décidable). -/
theorem spa_truthful_pos_three :
    0 < interimU1 (spa 3) ⟨2, by decide⟩
        (spaTruthful1 3 ⟨2, by decide⟩) (spaTruthful2 3) := by
  decide

-- Valeur concrète de l'utilité interim du type haut (valuations {0,1,2}) :
#eval interimU1 (spa 3) (2 : Fin 3) (spaTruthful1 3 (2 : Fin 3)) (spaTruthful2 3)


/-- Le type haut garde une rente d'information strictement positive (instance concrète décidable). -/
theorem spa_truthful_pos_three :
    0 < interimU1 (spa 3) ⟨2, by decide⟩
        (spaTruthful1 3 ⟨2, by decide⟩) (spaTruthful2 3) := by
  decide

-- Valeur concrète de l'utilité interim du type haut (valuations {0,1,2}) :
#eval interimU1 (spa 3) (2 : Fin 3) (spaTruthful1 3 (2 : Fin 3)) (spaTruthful2 3)
─────▶  6

--% env 7

Raw input:
{"cmd": "/-- Le type haut garde une rente d'information strictement positive (instance concr\u00e8te d\u00e9cidable). -/\ntheorem spa_truthful_pos_three :\n    0 < interimU1 (spa 3) \u27e82, by decide\u27e9\n        (spaTruthful1 3 \u27e82, by decide\u27e9) (spaTruthful2 3) := by\n  decide\n\n-- Valeur concr\u00e8te de l'utilit\u00e9 interim du type haut (valuations {0,1,2}) :\n#eval interimU1 (spa 3) (2 : Fin 3) (spaTruthful1 3 (2 : Fin 3)) (spaTruthful2 3)\n", "env": 6}
Raw output:
{"messages":
 [{"severity": "info",
   "pos": {"line": 8, "column": 0},
   "endPos": {"line": 8, "column": 5},
   "data": "6"}],
 "env": 7}

## 5. Récapitulatif et résultats complémentaires du lake

Ce notebook a **restaté et exécuté** le résultat central — le **théorème de Vickrey** (`spa_truthful_bne`, valable pour tout `n`, 0 sorry) — dans le kernel `lean4-wsl`. Le lake `lean_game_defs_ext/Bayesian` prouve en outre, également **sans sorry** :

- **`isBNE_scaleW`** (`BNE.lean`) : le statut de BNE est **invariant par remise à l'échelle du prior** (indépendance de normalisation — les poids `Nat` représentent toute mesure de probabilité proportionnelle).
- **Principe de déviation à un coup** (`exAnteU1/2_le_of_interim_best`, `bne_exAnte`) : l'optimalité interim implique l'optimalité ex-ante contre toute déviation contingente au type.
- **Monotonie de Blackwell** (`Information.lean`, `valueSignal_mono`) : pour un décideur seul, un signal plus fin (`σ = h ∘ τ`) vaut au moins autant — **l'information ne nuit jamais à un joueur solitaire** (mais peut nuire strictement dans un *jeu*, cf. `InfoGames.lean`).
- **Enchère au premier prix** (`Auction.lean`) : l'enchère honnête n'y est **pas** un BNE (`truthful_not_bne_two/three`).

Ces résultats se vérifient par la compilation du lake ci-dessus (`lake build Bayesian`, 0 sorry). Le pattern *self-contained* de ce notebook (restate inline, kernel élabore les vrais énoncés) est exactement celui des notebooks `2b/4b/8b/15b`.

### Références

- Vickrey, W. (1961). *Counterspeculation, Auctions, and Competitive Sealed Tenders*. Journal of Finance, 16(1), 8–37.
- Harsanyi, J. C. (1967). *Games with Incomplete Information Played by "Bayesian" Players*. Management Science, 14(3).
- Blackwell, D. (1951). *Comparison of Experiments*. Proc. 2nd Berkeley Symp. Math. Statist. Probab.

### Prochaine étape

L'extension à l'import direct d'un lake depuis le kernel `lean4-wsl` (prelude `NotebookCtx`, lancement via lake-env) est étudiée en side-track sur l'issue #4251 (Part of #4038).
